In [2]:
import torch
import torch.nn as nn

In [3]:
vgg16 =[64,64,'M',128,128,'M',256,256,256,'M',512,512,512,'M',512,512,512,'M']
vgg19=[64,64,'M',128,128,'M',256,256,256,256,'M',512,512,512,512,'M',512,512,512,512,'M']

In [6]:
# imge = torch.randn(batch_size,3,224,224)

class VGG(nn.Module):
    def __init__(self,in_channels=3,num_classes=100):
        super(VGG,self).__init__()
        self.in_channels=in_channels
        self.layers=self._make_layers(vgg16)

        self.fc=nn.Sequential(
            nn.Linear(512*7*7,4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096,4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096,num_classes)
        )


    def _make_layers(self,architecture):
        layers=[]
        for x in architecture:
            if type(x)==int:
                current_layer=nn.Sequential(
                    nn.Conv2d(self.in_channels,x,kernel_size=3,padding=1),
                    nn.BatchNorm2d(x),
                    nn.ReLU(inplace=True)
                )
                layers.append(current_layer)
                self.in_channels=x

            elif x=='M':
                layers.append(nn.MaxPool2d(kernel_size=2,stride=2))
        return nn.Sequential(*layers)

    def forward(self,x):
        res=self.layers(x)
        res=res.view(res.size(0),-1)
        res=self.fc(res)
        return res

In [10]:
x=torch.randn(32,3,224,224)
model=VGG()
y=model(x)

In [11]:
y.shape

torch.Size([32, 100])